# Noisy VQE Figures for Slides

Visualization notebook for synthetic-noise VQE caches. It starts with H2 only and can be compared with the noiseless statevector notebook.


## Setup


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.visualization.vqe_plots import (
    best_vqe_by_point,
    chemical_accuracy_config_table,
    ensure_figures_dir,
    load_latest_fci_curves,
    load_latest_vqe_results,
    plot_ansatz_comparison,
    plot_chemical_accuracy_rate,
    plot_dissociation_curve,
    plot_optimizer_comparison,
    plot_runtime_by_configuration,
    save_figure,
)

pd.set_option("display.max_columns", None)
output_dir = ensure_figures_dir("outputs/figures/slides_noisy")


## Load Latest Noisy Data


In [ ]:
all_vqe_df = load_latest_vqe_results()
fci_df = load_latest_fci_curves(strategy="densest")

noisy_df = all_vqe_df[
    (all_vqe_df["molecule"] == "H2")
    & (all_vqe_df["run_label"].astype(str).str.startswith("noisy_synthetic"))
].copy()

statevector_df = all_vqe_df[
    (all_vqe_df["molecule"] == "H2")
    & (all_vqe_df["run_label"].fillna("statevector").astype(str).str.startswith("statevector"))
].copy()

print(f"Noisy rows: {len(noisy_df)}")
print(f"Statevector rows: {len(statevector_df)}")
noisy_df[["molecule", "basis", "ansatz", "optimizer", "success", "source_path"]].drop_duplicates().sort_values(["basis", "ansatz", "optimizer"])


## Statevector baseline comparison

Use the best successful noisy and noiseless VQE point for each distance to estimate the practical noise penalty.


In [ ]:
best_noisy_compare = best_vqe_by_point(noisy_df).assign(mode="Aer noisy")
best_statevector_compare = best_vqe_by_point(statevector_df).assign(mode="Statevector")

comparison_df = best_noisy_compare.merge(
    best_statevector_compare,
    on=["molecule", "basis", "distance"],
    suffixes=("_noisy", "_statevector"),
)
comparison_df["noise_penalty_kcal_mol"] = (
    comparison_df["abs_error_kcal_mol_noisy"] - comparison_df["abs_error_kcal_mol_statevector"]
)

comparison_cols = [
    "molecule", "basis", "distance",
    "ansatz_noisy", "reps_noisy", "optimizer_noisy",
    "abs_error_kcal_mol_noisy",
    "ansatz_statevector", "reps_statevector", "optimizer_statevector",
    "abs_error_kcal_mol_statevector", "noise_penalty_kcal_mol",
]
comparison_df[comparison_cols].sort_values(["basis", "distance"])


In [ ]:
for basis in sorted(set(noisy_df["basis"]).intersection(statevector_df["basis"])):
    fig, ax = plt.subplots(figsize=(8, 5))
    for label, df, marker in [
        ("Statevector", best_statevector_compare, "o"),
        ("Aer noisy", best_noisy_compare, "s"),
    ]:
        subset = df[df["basis"] == basis].sort_values("distance")
        if subset.empty:
            continue
        ax.plot(
            subset["distance"],
            subset["abs_error_kcal_mol"],
            marker=marker,
            linewidth=2,
            label=label,
        )
    ax.axhline(1.0, color="#7a003c", linestyle=":", linewidth=2, label="Precis?o qu?mica (1 kcal/mol)")
    ax.set_title(f"Impacto do ru?do sint?tico - H2 ({basis})")
    ax.set_xlabel("Dist?ncia (?)")
    ax.set_ylabel("Erro absoluto (kcal/mol)")
    ax.grid(True, alpha=0.25)
    ax.legend()
    path = save_figure(fig, output_dir, f"noisy_vs_statevector_H2_{basis}.png")
    print(path)
    plt.show()


## Best Noisy Results by Point


In [ ]:
best_noisy_df = best_vqe_by_point(noisy_df)
best_noisy_df[[
    "molecule", "basis", "distance", "ansatz", "reps", "optimizer",
    "energy", "reference_energy", "abs_error_kcal_mol",
    "within_chemical_accuracy", "total_experiment_s",
]].sort_values(["basis", "distance"])


## Noisy Configuration DataFrame


In [ ]:
noisy_config_df = chemical_accuracy_config_table(noisy_df, only_accurate=False, statevector_only=False)

if noisy_config_df.empty:
    display(pd.DataFrame({
        "message": ["No successful noisy VQE rows were found for the configuration summary."],
        "noisy_rows": [len(noisy_df)],
    }))
else:
    display(noisy_config_df.sort_values([
        "molecule", "basis", "reached_chemical_accuracy", "accuracy_rate", "mean_error_kcal_mol"
    ], ascending=[True, True, False, False, True]))


## Noisy Dissociation Curves


In [ ]:
for molecule, basis in [("H2", "sto-3g"), ("H2", "6-31g")]:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_dissociation_curve(fci_df, noisy_df, molecule=molecule, basis=basis, ax=ax)
    path = save_figure(fig, output_dir, f"noisy_dissociacao_{molecule}_{basis}.png")
    print(path)
    plt.show()


## Noisy Ansatz Comparison


In [ ]:
for molecule, basis, optimizer in [("H2", "sto-3g", "cobyla"), ("H2", "6-31g", "cobyla")]:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_ansatz_comparison(noisy_df, molecule=molecule, basis=basis, optimizer=optimizer, ax=ax)
    path = save_figure(fig, output_dir, f"noisy_ansatz_{molecule}_{basis}_{optimizer}.png")
    print(path)
    plt.show()


## Noisy Optimizer Comparison


In [ ]:
for molecule, basis, ansatz in [("H2", "sto-3g", "efficient_su2"), ("H2", "6-31g", "efficient_su2")]:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_optimizer_comparison(noisy_df, molecule=molecule, basis=basis, ansatz=ansatz, ax=ax)
    path = save_figure(fig, output_dir, f"noisy_otimizadores_{molecule}_{basis}_{ansatz}.png")
    print(path)
    plt.show()


## Chemical Accuracy and Runtime


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot_chemical_accuracy_rate(noisy_df, ax=ax)
path = save_figure(fig, output_dir, "noisy_taxa_precisao_quimica.png")
print(path)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
plot_runtime_by_configuration(noisy_df, ax=ax)
path = save_figure(fig, output_dir, "noisy_tempo_medio_configuracao.png")
print(path)
plt.show()


## Generated Figures


In [ ]:
for path in sorted(output_dir.glob("*.png")):
    print(path)
